# StreetFoodAI: Exploratory Data Analysis & Nutrition Inspection

This notebook demonstrates local data exploration, class configuration inspection, nutritional composition distributions, and inference testing.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Load Nutritional Database
with open("../ml/config/nutrition_db.json", "r", encoding="utf-8") as f:
    nut_data = json.load(f)

foods = nut_data["foods"]
records = []
for name, info in foods.items():
    if name == "Other / Unknown Food":
        continue
    reg = info["servings"]["Regular"]
    records.append({
        "Food": name,
        "Weight (g)": reg["serving_weight_g"],
        "Calories (kcal)": reg["calories"],
        "Protein (g)": reg["protein"],
        "Carbs (g)": reg["carbohydrates"],
        "Fat (g)": reg["fat"],
        "Fiber (g)": reg["fiber"]
    })

df = pd.DataFrame(records)
df.sort_values(by="Calories (kcal)", ascending=False)

In [ ]:
# 2. Plot Caloric Density Comparison
plt.figure(figsize=(12, 6))
sorted_df = df.sort_values(by="Calories (kcal)", ascending=True)
plt.barh(sorted_df["Food"], sorted_df["Calories (kcal)"], color="#f97316", edgecolor="#c2410c")
plt.title("Estimated Energy per Regular Street Food Serving (kcal)", fontsize=14, fontweight="bold")
plt.xlabel("Calories (kcal)")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Test Local Predictor with Sample Image
from PIL import Image
from ml.inference.predictor import StreetFoodPredictor

predictor = StreetFoodPredictor(model_name="mobilenet_v3_large")
sample_path = Path("../data/raw/sample_test/vada_pav.jpg")

if sample_path.exists():
    img = Image.open(sample_path)
    res = predictor.predict(img)
    print(f"Predicted Food: {res['predicted_food']} ({res['confidence']*100:.1f}%)")
    print(f"Entropy: {res['entropy']:.3f} nats, Uncertain: {res['is_uncertain']}")